In [0]:
create or refresh streaming table stream_orders_bronze
as 
select * from STREAM read_files('/Volumes/kishore/cap_db/cap_files/stream_data/',
                        format =>'json')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4586528826745810>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "create or refresh streaming table stream_orders_bronze\nas \nselect * from read_files('/Volumes/kishore/cap_db/cap_files/stream_data/',\n                        format =>'json')\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbrunti

In [0]:
create or refresh live table stream_restaurants_bronze 
as 
select * from read_files('/FileStore/csv_files/dimensions/dim_restaurants.csv',
format =>'csv',header =>true,inferSchema => true)

In [0]:
create or refresh live table stream_items_bronze 
as 
select * from read_files('/FileStore/csv_files/dimensions/dim_menu_items.csv',
format =>'csv',header =>true,inferSchema => true)

In [0]:
create or refresh live table stream_customers_bronze 
as 
select * from read_files('/FileStore/csv_files/dimensions/dim_customers.csv',
format =>'csv',header =>true,inferSchema => true)

In [0]:
create or refresh live table stream_location_bronze 
as 
select * from read_files('/FileStore/csv_files/dimensions/dim_locations.csv',
format =>'csv',header =>true,inferSchema => true)

In [0]:
create or refresh live table stream_agents_bronze 
as 
select * from read_files('/FileStore/csv_files/dimensions/dim_delivery_agents.csv',
format =>'csv',header =>true,inferSchema => true)

In [0]:
create or refresh streaming live table cleaned_orders (
    constraint vaild_order_id expect(order_id is not null) on violation drop row,
    constraint valid_customer expect(customer_id is not null) on violation drop row,
    constraint valid_items expect(size(items_ordered) >0) on violation drop row
) 
as 
select order_id,timestamp as order_ts,customer_id,restaurant_id,agent_id,delivery_location_id,items_ordered,explode(items_ordered) as item,
total_amount,tip,payment_method,order_status
from stream(live.stream_orders_bronze)

In [0]:
create or refresh streaming table orders_silver(
    constraint valid_restaurant_name expect(restaurant_name is not null) on violation drop row,
    constraint valid_agent expect(agent_name is not null) on violation drop row,
    constraint valid_item expect(item_id is not null) on violation drop row 
) as
select o.order_id,o.order_ts,o.customer_id,
c.name as customer_name,c.email,o.restaurant_id,r.name as restaurant_name
,r.cuisines,r.rating as restaurant_rating,
o.agent_id,a.name as agent_name,a.rating as agent_rating,o.delivery_location_id,
l.city,o.item.item_id as item_id,i.item_name ,i.category,
o.item.price as item_price,
o.item.quantity as item_quantity,
o.total_amount,o.tip,o.payment_method,o.order_status
from stream(live.cleaned_orders) o left join live.stream_customers_bronze c on o.customer_id = c.customer_id
left join live.stream_restaurants_bronze r on o.restaurant_id = r.restaurant_id
left join live.stream_agents_bronze a on o.agent_id = a.agent_id
left join live.stream_items_bronze i on o.item.item_id = i.item_id
left join live.stream_location_bronze l on o.delivery_location_id = l.location_id

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4586528826745818>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'create or refresh streaming table orders_silver()\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, line, cell)
    210 except BaseException as e:
    211     self.driver_acti

In [0]:
create or refresh streaming table orders_with_agents 
as 
select o.order_id,o.order_ts,o.agent_id,a.name as agent_name,a.rating as agent_rating,o.tip
from stream(live.cleaned_orders) o 
left join stream_agents_bronze a on o.agent_id = a.agent_id

In [0]:
create or refresh streaming table orders_with_items 
as 
select 
    o.order_id,o.order_ts,o.restaurant_id,exploded.item_id,i.item_name,i.category,
    exploded.price,exploded.quantity
from (select order_id,timestamp as order_ts,restaurant_id,explode(items_ordered) as exploded from stream(live.stream_orders_bronze)) o 
left join stream_items_bronze i on o.exploded.item_id = i.item_id

In [0]:
create or refresh streaming table orders_with_location 
as 
select 
    o.order_id,o.order_ts,o.delivery_location_id,l.city,l.state,l.area
 from stream(live.cleaned_orders) o 
left join stream_location_bronze l 
on o.delivery_location_id = l.location_id

In [0]:
-- Gold Layer Queries 
-- 1. Total orders per restaurant 
create live table total_orders_per_restaurant as 
select restaurant_name,count(distinct order_id) as total_orders
from orders_silver
group by restaurant_name 

In [0]:
-- 2.Top selling items
create live table top_selling_items as 
select item_name,sum(item_quantity) as total_qty_sold from orders_silver
group by item_name 
order by total_qty_sold desc

In [0]:
-- 3.Avg delivery fee and tip by city 
create live table avg_fees_by_city as 
select city,avg(restaurant_rating) as avg_restaurant_rating,avg(tip) as avg_tip from orders_silver group by city

In [0]:
-- 4. Agent performance
create live table agent_performance as 
select agent_name,count(distinct order_id) as orders_handled,avg(agent_rating) as avg_rating from orders_silver group by agent_name order by orders_handled desc

In [0]:
-- 5. Revenue trend
create live table daily_revenue_trend 
as 
select date(order_ts) as order_date,sum(total_amount) as total_revenue from live.orders_silver group by order_date order by order_date

In [0]:
-- 6. Customer lifetime value 
create live table customer_ltv as 
select customer_id,customer_name,sum(total_amount) as lifetime_spend from orders_silver
group by customer_id,customer_name
order by lifetime_spend desc 

In [0]:
create or refresh streaming table orders_with_customers 
as 
select 
    o.order_id,o.customer_id,c.name as customer_name,c.location_id as customer_location_id , o.total_amount,o.tip,o.payment_method,o.order_status
 from stream(live.cleaned_orders) o 
left join live.stream_customers_bronze c on o.customer_id = c.customer_id

In [0]:
create live table customer_order_counts as 
select customer_name ,count(*) as total_orders 
from orders_with_customers
group by customer_name